In [ ]:
"""Run in Kaggle after attaching the official 2026task2 competition data."""
from pathlib import Path
from collections import Counter
import csv
import hashlib
import json
from PIL import Image

ROOT = Path('/kaggle/input/competitions/2026task2')
if not ROOT.exists():
    ROOT = Path('/kaggle/input/2026task2')
if not ROOT.exists():
    raise FileNotFoundError('Attach the official 2026task2 input before running.')
LABELS = {'angry': 0, 'anger': 0, 'fear': 1, 'fearful': 1,
          'happy': 2, 'happiness': 2, 'sad': 3, 'sadness': 3,
          'surprise': 4, 'surprised': 4, 'neutral': 5}
report = {'competition': '2026task2', 'root': str(ROOT),
          'label_map': LABELS, 'directories': [], 'csv_files': [],
          'images_by_directory': {}, 'image_sizes_modes': {},
          'unreadable_images': [], 'exact_duplicate_groups': []}
report['directories'] = sorted(str(p.relative_to(ROOT)) for p in ROOT.rglob('*')
                               if p.is_dir())
for p in sorted(ROOT.rglob('*.csv')):
    with p.open(newline='', encoding='utf-8-sig') as handle:
        reader = csv.DictReader(handle)
        rows = list(reader)
        report['csv_files'].append({'file': str(p.relative_to(ROOT)),
                                   'columns': reader.fieldnames,
                                   'rows': len(rows), 'first_rows': rows[:3]})
images = sorted(p for p in ROOT.rglob('*')
                if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'})
counts, sizes, hashes = Counter(), Counter(), {}
for p in images:
    relative = str(p.relative_to(ROOT))
    counts[str(p.parent.relative_to(ROOT))] += 1
    try:
        with Image.open(p) as im:
            im.load()
            sizes[str((im.size, im.mode))] += 1
            # Compare actual pixels, since JPEG container bytes can differ.
            rgb = im.convert('RGB')
            digest = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()
            hashes.setdefault(digest, []).append(relative)
    except Exception as exc:
        report['unreadable_images'].append({'file': relative, 'error': str(exc)})
report['images_by_directory'] = dict(sorted(counts.items()))
report['image_sizes_modes'] = dict(sizes)
report['total_images'] = len(images)
report['exact_duplicate_groups'] = [paths for paths in hashes.values() if len(paths) > 1]
out = Path('/kaggle/working/task2_data_audit.json')
out.write_text(json.dumps(report, indent=2, ensure_ascii=False))
print(json.dumps({k: v for k, v in report.items()
                  if k not in {'directories', 'exact_duplicate_groups'}},
                 indent=2, ensure_ascii=False))
print('Exact duplicate groups:', len(report['exact_duplicate_groups']))
print('Saved audit:', out)


In [ ]:
"""Official-data-only FER baseline; execute in a GPU Kaggle notebook."""
from pathlib import Path
from collections import Counter
import copy
import hashlib
import json
import random
import time
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

SEED = 42
SIZE = 112
EPOCHS = 35
BATCH = 128
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(2)
torch.backends.cudnn.benchmark = True
DEVICE = torch.device('cuda:0')
assert torch.cuda.is_available(), 'GPU required for this training run'
ROOT = Path('/kaggle/input/competitions/2026task2')
if not ROOT.exists():
    ROOT = Path('/kaggle/input/2026task2')
LABELS = {'angry': 0, 'anger': 0, 'fear': 1, 'fearful': 1,
          'happy': 2, 'happiness': 2, 'sad': 3, 'sadness': 3,
          'surprise': 4, 'surprised': 4, 'neutral': 5}
template = pd.read_csv(ROOT / 'submission.csv')
assert template.columns.tolist() == ['ID', 'Emotion']
assert template.ID.is_unique
test_ids = set(template.ID.astype(str))
train_paths, arrays, labels, groups = [], [], [], []
test_paths = {}
for p in sorted(ROOT.rglob('*.jpg')):
    if p.name in test_ids:
        assert p.name not in test_paths, f'Ambiguous test ID: {p.name}'
        test_paths[p.name] = p
    elif p.parent.name.lower() in LABELS:
        with Image.open(p) as im:
            a = np.array(im.convert('L'))
        train_paths.append(str(p.relative_to(ROOT)))
        arrays.append(a)
        labels.append(LABELS[p.parent.name.lower()])
        groups.append(hashlib.sha256(str(a.shape).encode() + a.tobytes()).hexdigest())
    else:
        raise ValueError(f'Unrecognized image location: {p.relative_to(ROOT)}')
assert set(test_paths) == test_ids
assert set(labels) == set(range(6))
Y = np.array(labels)
tr, va = next(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(arrays, Y, groups))
assert not set(np.array(groups)[tr]) & set(np.array(groups)[va])
print('DATA', len(tr), len(va), len(test_ids), Counter(labels), flush=True)
test_arrays = []
for name in template.ID.astype(str):
    with Image.open(test_paths[name]) as im:
        test_arrays.append(np.array(im.convert('L')))
norm = T.Normalize([.485, .456, .406], [.229, .224, .225])
train_tf = T.Compose([T.Resize((SIZE, SIZE)), T.RandomHorizontalFlip(),
                      T.RandomAffine(10, translate=(.06, .06), scale=(.9, 1.05)),
                      T.ColorJitter(brightness=.12, contrast=.12),
                      T.ToTensor(), norm, T.RandomErasing(p=.1, scale=(.01,.04))])
eval_tf = T.Compose([T.Resize((SIZE, SIZE)), T.ToTensor(), norm])
class Faces(Dataset):
    def __init__(self, images, targets, tf):
        self.images, self.targets, self.tf = images, targets, tf
    def __len__(self):
        return len(self.images)
    def __getitem__(self, i):
        return self.tf(Image.fromarray(self.images[i]).convert('RGB')), int(self.targets[i])
def loader(indices, train=False):
    return DataLoader(Faces([arrays[i] for i in indices], Y[indices], train_tf if train else eval_tf),
                      batch_size=BATCH, shuffle=train, num_workers=2, pin_memory=True)
train_dl, valid_dl = loader(tr, True), loader(va)
test_dl = DataLoader(Faces(test_arrays, np.zeros(len(test_arrays)), eval_tf),
                     batch_size=BATCH, num_workers=2, pin_memory=True)
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 6)
model = model.to(DEVICE)
opt = torch.optim.AdamW([{'params': [p for n,p in model.named_parameters() if not n.startswith('fc.')], 'lr': 3e-4},
                        {'params': model.fc.parameters(), 'lr': 1e-3}], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS, eta_min=1e-6)
criterion = nn.CrossEntropyLoss(label_smoothing=.03)
scaler = torch.amp.GradScaler('cuda')
@torch.inference_mode()
def predict(dl, flip=True):
    model.eval()
    result = []
    for x,_ in dl:
        x=x.to(DEVICE,non_blocking=True)
        with torch.autocast('cuda'):
            p=model(x).softmax(1)
            if flip:
                p=(p+model(x.flip(-1)).softmax(1))/2
        result.append(p.float().cpu().numpy())
    return np.concatenate(result)
history, best, stale, best_state, best_epoch = [], -1., 0, None, None
started = time.time()
for epoch in range(EPOCHS):
    model.train()
    loss_sum, correct, seen = 0., 0, 0
    for x,y in train_dl:
        x,y=x.to(DEVICE,non_blocking=True),y.to(DEVICE,non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast('cuda'):
            logits=model(x)
            loss=criterion(logits,y)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(),5.)
        scaler.step(opt)
        scaler.update()
        loss_sum+=loss.item()*len(y)
        correct+=(logits.argmax(1)==y).sum().item()
        seen+=len(y)
    scheduler.step()
    probs=predict(valid_dl)
    acc=accuracy_score(Y[va],probs.argmax(1))
    row={'epoch':epoch+1,'train_loss':loss_sum/seen,'train_accuracy':correct/seen,
         'valid_accuracy':acc,'seconds':time.time()-started}
    history.append(row)
    pd.DataFrame(history).to_csv('/kaggle/working/fer_training_history.csv',index=False)
    print('EPOCH',json.dumps(row),flush=True)
    if acc>best:
        best,best_epoch,stale=acc,epoch+1,0
        best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        torch.save({'state_dict':best_state,'epoch':best_epoch,'valid_accuracy':best,
                    'model':'resnet18','size':SIZE,'label_map':LABELS},'/kaggle/working/fer_resnet18_best.pt')
    else:
        stale+=1
    if stale>=8 and epoch>=15:
        break
model.load_state_dict(best_state)
vp=predict(valid_dl)
tp=predict(test_dl)
np.save('/kaggle/working/fer_valid_indices.npy',va)
np.save('/kaggle/working/fer_valid_probs.npy',vp)
np.save('/kaggle/working/fer_test_probs.npy',tp)
pd.DataFrame({'path':train_paths,'label':Y,'group':groups}).to_csv('/kaggle/working/fer_training_index.csv',index=False)
submission=template.copy()
submission['Emotion']=tp.argmax(1).astype(int)
assert submission.ID.equals(template.ID) and len(submission)==len(test_ids)
assert not submission.isna().any().any() and submission.Emotion.between(0,5).all()
out=Path('/kaggle/working/submission_fer_resnet18.csv')
submission.to_csv(out,index=False)
manifest={'architecture':'resnet18','pretraining':'ImageNet only','seed':SEED,
          'image_size':SIZE,'epochs':len(history),'best_epoch':best_epoch,
          'validation_accuracy':float(best),'train_count':len(Y),'validation_count':len(va),
          'test_count':len(test_ids),'label_map':LABELS,
          'validation_split':'StratifiedGroupKFold, pixel duplicate groups, first of five folds',
          'classification_report':classification_report(Y[va],vp.argmax(1),output_dict=True),
          'confusion_matrix':confusion_matrix(Y[va],vp.argmax(1)).tolist(),
          'submission_sha256':hashlib.sha256(out.read_bytes()).hexdigest(),
          'training_data':'official competition training labels only',
          'runtime_seconds':time.time()-started}
Path('/kaggle/working/fer_experiment_manifest.json').write_text(json.dumps(manifest,indent=2))
print('FINISHED',json.dumps(manifest),flush=True)


In [ ]:
"""Second Kaggle cell: compare EfficientNet-B2 on the same grouped validation split.

Requires the arrays and train/validation indices from train_fer.py in memory.
No test labels are used. Ensemble weights are selected only on validation data.
"""
import copy
assert 'arrays' in globals() and 'tr' in globals() and 'va' in globals()
baseline_vp = np.load('/kaggle/working/fer_valid_probs.npy')
baseline_tp = np.load('/kaggle/working/fer_test_probs.npy')
SIZE2, EPOCHS2 = 160, 35
torch.manual_seed(43)
torch.cuda.manual_seed_all(43)
train_tf2 = T.Compose([T.Resize((SIZE2, SIZE2)),
                       T.RandomResizedCrop(SIZE2, scale=(.8, 1.), ratio=(.95, 1.05)),
                       T.RandomHorizontalFlip(),
                       T.RandomAffine(10, translate=(.06, .06), scale=(.9, 1.05)),
                       T.ColorJitter(brightness=.12, contrast=.12),
                       T.ToTensor(), norm, T.RandomErasing(p=.1, scale=(.01, .04))])
eval_tf2 = T.Compose([T.Resize((SIZE2, SIZE2)), T.ToTensor(), norm])
def loader2(images, targets, tf, train=False):
    return DataLoader(Faces(images, targets, tf), batch_size=96,
                      shuffle=train, num_workers=2, pin_memory=True)
td2 = loader2([arrays[i] for i in tr], Y[tr], train_tf2, True)
vd2 = loader2([arrays[i] for i in va], Y[va], eval_tf2)
xd2 = loader2(test_arrays, np.zeros(len(test_arrays)), eval_tf2)
m2 = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.DEFAULT)
m2.classifier = nn.Sequential(nn.Dropout(.3), nn.Linear(m2.classifier[1].in_features, 6))
m2 = m2.to(DEVICE)
ema2 = copy.deepcopy(m2).eval()
for p in ema2.parameters():
    p.requires_grad_(False)
opt2 = torch.optim.AdamW([{'params': m2.features.parameters(), 'lr': 2e-4},
                         {'params': m2.classifier.parameters(), 'lr': 8e-4}], weight_decay=1e-4)
sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, EPOCHS2, eta_min=1e-6)
scaler2 = torch.amp.GradScaler('cuda')
@torch.inference_mode()
def predict2(net, dl):
    net.eval()
    result=[]
    for x,_ in dl:
        x=x.to(DEVICE,non_blocking=True)
        with torch.autocast('cuda'):
            p=(net(x).softmax(1)+net(x.flip(-1)).softmax(1))/2
        result.append(p.float().cpu().numpy())
    return np.concatenate(result)
best2, stale2, hist2 = -1., 0, []
start2=time.time()
for epoch2 in range(EPOCHS2):
    m2.train()
    ls, cor, n=0.,0,0
    for x,y in td2:
        x,y=x.to(DEVICE,non_blocking=True),y.to(DEVICE,non_blocking=True)
        opt2.zero_grad(set_to_none=True)
        lam=1.
        perm=torch.arange(len(y),device=DEVICE)
        if random.random()<.5:
            lam=float(np.random.beta(.2,.2))
            perm=torch.randperm(len(y),device=DEVICE)
            x=lam*x+(1-lam)*x[perm]
        with torch.autocast('cuda'):
            z=m2(x)
            loss=lam*criterion(z,y)+(1-lam)*criterion(z,y[perm])
        scaler2.scale(loss).backward()
        scaler2.unscale_(opt2)
        nn.utils.clip_grad_norm_(m2.parameters(),5.)
        scaler2.step(opt2)
        scaler2.update()
        with torch.no_grad():
            for ep,mp in zip(ema2.parameters(),m2.parameters()):
                ep.mul_(.995).add_(mp,alpha=.005)
            for eb,mb in zip(ema2.buffers(),m2.buffers()):
                eb.copy_(mb)
        ls+=loss.item()*len(y)
        cor+=lam*(z.argmax(1)==y).sum().item()+(1-lam)*(z.argmax(1)==y[perm]).sum().item()
        n+=len(y)
    sch2.step()
    pp2=predict2(ema2,vd2)
    ac2=accuracy_score(Y[va],pp2.argmax(1))
    row2={'epoch':epoch2+1,'train_loss':ls/n,'train_mixup_accuracy':cor/n,
          'valid_accuracy':ac2,'seconds':time.time()-start2}
    hist2.append(row2)
    pd.DataFrame(hist2).to_csv('/kaggle/working/fer_effnet_history.csv',index=False)
    print('EFFNET_EPOCH',json.dumps(row2),flush=True)
    if ac2>best2:
        best2,stale2,best_ep2=ac2,0,epoch2+1
        state2={k:v.detach().cpu().clone() for k,v in ema2.state_dict().items()}
        torch.save({'state_dict':state2,'epoch':best_ep2,'validation_accuracy':best2,
                    'model':'efficientnet_b2','image_size':SIZE2,'label_map':LABELS},
                   '/kaggle/working/fer_effnet_b2_best.pt')
    else:
        stale2+=1
    if stale2>=8 and epoch2>=15:
        break
ema2.load_state_dict(state2)
v2=predict2(ema2,vd2)
t2=predict2(ema2,xd2)
np.save('/kaggle/working/fer_effnet_valid_probs.npy',v2)
np.save('/kaggle/working/fer_effnet_test_probs.npy',t2)
weights=[0.,.25,.5,.75,1.]
scores={str(w):float(accuracy_score(Y[va],((1-w)*baseline_vp+w*v2).argmax(1))) for w in weights}
weight=max(weights,key=lambda w:scores[str(w)])
chosen=(1-weight)*baseline_tp+weight*t2
selected=template.copy()
selected['Emotion']=chosen.argmax(1).astype(int)
assert selected.ID.equals(template.ID) and not selected.isna().any().any()
assert selected.Emotion.between(0,5).all()
out2=Path('/kaggle/working/submission_fer_selected.csv')
selected.to_csv(out2,index=False)
result2={'model':'ResNet18 / EfficientNet-B2 EMA comparison',
         'effnet_best_epoch':best_ep2,'effnet_validation_accuracy':float(best2),
         'ensemble_validation_scores':scores,'efficientnet_weight':weight,
         'selected_validation_accuracy':scores[str(weight)],'seed_effnet':43,
         'training_data':'official competition train labels only; ImageNet pretraining',
         'rows':len(selected),'sha256':hashlib.sha256(out2.read_bytes()).hexdigest(),
         'seconds':time.time()-start2}
Path('/kaggle/working/fer_selected_manifest.json').write_text(json.dumps(result2,indent=2))
print('EFFNET_FINISHED',json.dumps(result2),flush=True)


In [ ]:
"""After validation selection, fine-tune on all official training images.

Validation labels now become training labels. Do not report any post-refit score
on this old validation split as an unbiased validation accuracy.
"""
assert 'result2' in globals() and 'state2' in globals()
all_base = DataLoader(Faces(arrays, Y, train_tf), batch_size=128,
                      shuffle=True, num_workers=2, pin_memory=True)
all_eff = DataLoader(Faces(arrays, Y, train_tf2), batch_size=96,
                     shuffle=True, num_workers=2, pin_memory=True)
full1 = copy.deepcopy(model).to(DEVICE)
full2 = copy.deepcopy(ema2).to(DEVICE)
for p in full1.parameters():
    p.requires_grad_(True)
for p in full2.parameters():
    p.requires_grad_(True)
def fit_full(net, dl, epochs, name, lr, mixup=False):
    avg=copy.deepcopy(net).eval()
    for p in avg.parameters():
        p.requires_grad_(False)
    optimizer=torch.optim.AdamW(net.parameters(),lr=lr,weight_decay=1e-4)
    schedule=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,epochs,eta_min=1e-6)
    amp=torch.amp.GradScaler('cuda')
    records=[]
    start=time.time()
    for e in range(epochs):
        net.train()
        total,loss_sum=0,0.
        for x,y in dl:
            x,y=x.to(DEVICE,non_blocking=True),y.to(DEVICE,non_blocking=True)
            lam=1.
            perm=torch.arange(len(y),device=DEVICE)
            if mixup and random.random()<.5:
                lam=float(np.random.beta(.2,.2))
                perm=torch.randperm(len(y),device=DEVICE)
                x=lam*x+(1-lam)*x[perm]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda'):
                z=net(x)
                loss=lam*criterion(z,y)+(1-lam)*criterion(z,y[perm])
            amp.scale(loss).backward()
            amp.unscale_(optimizer)
            nn.utils.clip_grad_norm_(net.parameters(),5.)
            amp.step(optimizer)
            amp.update()
            with torch.no_grad():
                for a,b in zip(avg.parameters(),net.parameters()):
                    a.mul_(.99).add_(b,alpha=.01)
                for a,b in zip(avg.buffers(),net.buffers()):
                    a.copy_(b)
            total+=len(y)
            loss_sum+=loss.item()*len(y)
        schedule.step()
        record={'model':name,'epoch':e+1,'loss':loss_sum/total,'seconds':time.time()-start}
        records.append(record)
        print('FULL_TRAIN',json.dumps(record),flush=True)
        pd.DataFrame(records).to_csv('/kaggle/working/'+name+'_full_history.csv',index=False)
    torch.save({'state_dict':avg.state_dict(),'full_train_images':len(Y),
                'epochs':epochs,'model':name,'label_map':LABELS},
               '/kaggle/working/'+name+'_full.pt')
    return avg
random.seed(44)
np.random.seed(44)
torch.manual_seed(44)
full1=fit_full(full1,all_base,5,'fer_resnet18',3e-5)
full2=fit_full(full2,all_eff,8,'fer_effnet_b2',4e-5,True)
tf1=predict2(full1,test_dl)
tf2=predict2(full2,xd2)
np.save('/kaggle/working/fer_resnet18_full_test_probs.npy',tf1)
np.save('/kaggle/working/fer_effnet_full_test_probs.npy',tf2)
w=result2['efficientnet_weight']
final_probs=(1-w)*tf1+w*tf2
final_csv=template.copy()
final_csv['Emotion']=final_probs.argmax(1).astype(int)
assert final_csv.ID.equals(template.ID) and final_csv.Emotion.between(0,5).all()
assert not final_csv.isna().any().any()
out_full=Path('/kaggle/working/submission_fer_full.csv')
final_csv.to_csv(out_full,index=False)
full_manifest={'method':'validation-selected models and ensemble weight; full-data low-rate fine-tuning',
               'train_count':len(Y),'resnet18_full_epochs':5,'effnet_b2_full_epochs':8,
               'ensemble_weight':w,'pre_refit_validation_accuracy':result2['selected_validation_accuracy'],
               'post_refit_validation_accuracy':None,'seed_refit':44,
               'training_data':'all official training labels only; ImageNet pretraining',
               'rows':len(final_csv),'sha256':hashlib.sha256(out_full.read_bytes()).hexdigest()}
Path('/kaggle/working/fer_full_manifest.json').write_text(json.dumps(full_manifest,indent=2))
print('FULL_FINISHED',json.dumps(full_manifest),flush=True)


In [ ]:
"""Compare inference resolutions using the untouched, pre-refit validation model.

Do not use the full-data model to score the old validation set.
"""
assert 'full2' in globals() and 'result2' in globals()
vp_base=np.load('/kaggle/working/fer_valid_probs.npy')
vp_eff=np.load('/kaggle/working/fer_effnet_valid_probs.npy')
tp_full_base=np.load('/kaggle/working/fer_resnet18_full_test_probs.npy')
tp_full_eff=np.load('/kaggle/working/fer_effnet_full_test_probs.npy')
val_candidates={160:vp_eff}
for resolution in [192,224]:
    tf=T.Compose([T.Resize((resolution,resolution)),T.ToTensor(),norm])
    vl=loader2([arrays[i] for i in va],Y[va],tf)
    val_candidates[resolution]=predict2(ema2,vl)
    print('RESOLUTION_VALIDATION',resolution,
          accuracy_score(Y[va],val_candidates[resolution].argmax(1)),flush=True)
    np.save('/kaggle/working/fer_effnet_valid_'+str(resolution)+'.npy',val_candidates[resolution])
options=[]
for resolution,probabilities in val_candidates.items():
    for weight in [0.,.25,.5,.75,1.]:
        score=float(accuracy_score(Y[va],((1-weight)*vp_base+weight*probabilities).argmax(1)))
        options.append({'resolution':resolution,'effnet_weight':weight,'validation_accuracy':score})
choice=max(options,key=lambda row:row['validation_accuracy'])
resolution=choice['resolution']
weight=choice['effnet_weight']
if resolution!=160 and weight>0:
    tf=T.Compose([T.Resize((resolution,resolution)),T.ToTensor(),norm])
    tl=loader2(test_arrays,np.zeros(len(test_arrays)),tf)
    tp_full_eff=predict2(full2,tl)
candidate=template.copy()
candidate['Emotion']=((1-weight)*tp_full_base+weight*tp_full_eff).argmax(1).astype(int)
assert candidate.ID.equals(template.ID) and candidate.Emotion.between(0,5).all()
assert not candidate.isna().any().any()
out_inference=Path('/kaggle/working/submission_fer_inference.csv')
candidate.to_csv(out_inference,index=False)
inference_record={'selection':'pre-refit grouped validation models only',
                  'choices':options,'selected':choice,'refit_validation_score':None,
                  'rows':len(candidate),'sha256':hashlib.sha256(out_inference.read_bytes()).hexdigest()}
Path('/kaggle/working/fer_inference_manifest.json').write_text(json.dumps(inference_record,indent=2))
print('INFERENCE_FINISHED',json.dumps(inference_record),flush=True)


In [ ]:
"""Known training-image consistency, using official training labels only.

Apply a deterministic exact-pixel match only when all matching training labels
agree. Ambiguous and near-blank matches keep the deep-model prediction.
No external data, test labels or approximate matching are used.
"""
from collections import defaultdict
train_labels_by_hash=defaultdict(set)
train_sources_by_hash=defaultdict(list)
for i,image in enumerate(arrays):
    if float(np.std(image))<=1.:
        continue
    digest=hashlib.sha256(str(image.shape).encode()+image.tobytes()).hexdigest()
    train_labels_by_hash[digest].add(int(Y[i]))
    train_sources_by_hash[digest].append(train_paths[i])
deep_candidate=pd.read_csv('/kaggle/working/submission_fer_inference.csv')
assert deep_candidate.ID.equals(template.ID)
consistent=deep_candidate.copy()
matches=[]
for i,image in enumerate(test_arrays):
    if float(np.std(image))<=1.:
        continue
    digest=hashlib.sha256(str(image.shape).encode()+image.tobytes()).hexdigest()
    known=train_labels_by_hash.get(digest,set())
    if len(known)!=1:
        continue
    training_label=next(iter(known))
    before=int(consistent.iloc[i]['Emotion'])
    consistent.at[i,'Emotion']=training_label
    matches.append({'ID':str(consistent.iloc[i]['ID']),
                    'deep_prediction':before,'official_training_label':training_label,
                    'changed':before!=training_label,'source_train_images':train_sources_by_hash[digest]})
assert consistent.ID.equals(template.ID) and consistent.Emotion.between(0,5).all()
assert not consistent.isna().any().any()
out_consistent=Path('/kaggle/working/submission_fer_consistent.csv')
consistent.to_csv(out_consistent,index=False)
consistency_record={'method':'deep ensemble plus exact-pixel consistency with unambiguous official training labels',
                    'matched_test_images':len(matches),'changed_predictions':sum(r['changed'] for r in matches),
                    'test_labels_used':False,'external_labels_used':False,
                    'unmatched_prediction_source':'validation-selected, full-data-trained deep models',
                    'rows':len(consistent),'sha256':hashlib.sha256(out_consistent.read_bytes()).hexdigest()}
Path('/kaggle/working/fer_consistency_manifest.json').write_text(json.dumps(consistency_record,indent=2))
Path('/kaggle/working/fer_consistency_matches.json').write_text(json.dumps(matches,indent=2))
print('CONSISTENCY_FINISHED',json.dumps(consistency_record),flush=True)
